In [1]:

import pandas as pd
import argparse

def read_dataframe(year, month):

    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet'

    df = pd.read_parquet(url)
    print(f'Loaded {len(df)} rows from {url}')

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    print(f'Resulting rows after filtering: {len(df)}')

    return df
    


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction import DictVectorizer
import mlflow
import pickle
import os



def train_and_log_model(df):   
    dv = DictVectorizer()
    lr = LinearRegression()

    categorical = ['PULocationID', 'DOLocationID']
    dicts = df[categorical].to_dict(orient='records')
    X = dv.fit_transform(dicts)
    y = df.duration.values
    lr.fit(X, y)

    y_pred = lr.predict(X)
    rmse = mean_squared_error(y, y_pred)**(1/2)

    print(f'Intercept: {lr.intercept_}')
    print(f'RMSE: {rmse}')

    with mlflow.start_run():
        mlflow.set_experiment('nyc-taxi-experiment')
        mlflow.set_tracking_uri('http://localhost:5000')
        mlflow.log_param("model_type", "linear_regression")
        mlflow.log_param("vectorizer", "DictVectorizer")
        mlflow.log_metric('rmse', rmse)

        # save model to a local folder
        os.makedirs('mlflow_artifact', exist_ok=True)
        model_path = 'mlflow_artifact/model.pkl'
        vd_path = 'mlflow_artifact/dv.pkl'

        with open(model_path, 'wb') as f_out:
            pickle.dump(lr, f_out)
        with open(vd_path, 'wb') as f_out:
            pickle.dump(dv, f_out)

        mlflow.log_artifact(model_path, artifact_path="model")
        mlflow.log_artifact(vd_path, artifact_path="dv.pkl")



    print('Model and artifacts logged to MLflow')

    return dv, lr



In [ ]:
df = read_dataframe(2023, 3)
dv, lr = train_and_log_model(df)

Intercept: 24.776359644078624
RMSE: 8.158681472091054
🏃 View run capricious-fox-287 at: http://localhost:5000/#/experiments/362584147077783893/runs/a5ef07ae4de74e138a83cbda9cdd918b
🧪 View experiment at: http://localhost:5000/#/experiments/362584147077783893
Model and artifacts logged to MLflow
